In [1]:
import numpy as np

from typing import Tuple

from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

I0000 00:00:1773447732.281452 1134704 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1773447732.333354 1134704 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773447733.425071 1134704 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
# 生成批大小 N=2, 通道 C=1, 高 H=4, 宽 W=4 的玩具数据
# 使用 1 到 32 的顺序整数，方便在经过卷积/池化后追踪数字的变化规律
x_toy = np.arange(1, 19).reshape(2, 1, 3, 3)
y_toy = np.array([[1, 0, 0], [0, 0, 1]])
# w_toy = np.arange(1, 19).reshape(2, 1, 3, 3)
w_toy = np.ones((2, 1, 2, 2))
b_toy = np.array([0, 1])

# print("玩具数据形状:", x_toy.shape)
# print("玩具数据:\n", x_toy)
# print()
# print("玩具权重:\n", w_toy)

In [3]:
# 加载数据
(x_train, t_train), (x_test, t_test) = mnist.load_data()

# 归一化并将数据形状重塑为 (批大小, 通道数, 高, 宽) 即 (N, C, H, W)
# 书中提到，MNIST 输入图像是 1 通道、高 28 像素、长 28 像素的 (1, 28, 28) 形状
# 并且 CNN 中各层间传递的数据是 4 维数据 [cite: 461, 589]
x_train = x_train.reshape(-1, 1, 28, 28) / 255.0
x_test = x_test.reshape(-1, 1, 28, 28) / 255.0

# 只取前 1000 个训练样本和 200 个测试样本用于学习
x_train = x_train[:1000]
t_train = t_train[:1000]
x_test = x_test[:200]
t_test = t_test[:200]

In [4]:
print(f'X_train: {x_train.shape}\n' \
      f't_train: {t_train.shape}\n' \
      f'X_test: {x_test.shape}\n' \
      f't_test: {t_test.shape}')

X_train: (1000, 1, 28, 28)
t_train: (1000,)
X_test: (200, 1, 28, 28)
t_test: (200,)


In [5]:
t_train_one_hot = to_categorical(t_train, 10)

计算输出特征图大小 $(OH, OW)$ 的公式如下 ：

$$OH = \frac{H + 2P - FH}{S} + 1$$

$$OW = \frac{W + 2P - FW}{S} + 1$$

其中：

* **$H$, $W$**: 输入数据的高和宽。
* **$FH$, $FW$**: 滤波器（卷积核）的高和宽。
* **$P$**: 填充 (Padding)，向输入数据周围填入固定数据的大小。主要是为了避免每次卷积后空间缩小。
* **$S$**: 步幅 (Stride)，应用滤波器位置的间隔。如果步幅变大，输出大小将会缩小。

通过上面两个公式，求得输出特征的尺寸

In [6]:
def calculate_output_size(H: int,
                          W: int,
                          FH: int,
                          FW: int,
                          P: int = 0,
                          S: int = 1) -> Tuple[int, int]:
    """计算卷积层（或池化层）输出特征图的空间尺寸（高和宽）

    Args:
    H (int): 输入数据的高度
    W (int): 输入数据的宽度
    FH (int): 滤波器的高度
    FW (int): 滤波器的宽度
    P (int, optional): 填充大小. 默认为 0.
    S (int, optional): 步幅大小. 默认为 1.

    Returns:
    Tuple[int, int]:
    - OH: 输出的高度
    - OW: 输出的宽度
    """

    OH = int((H + 2 * P - FH) / S + 1)
    OW = int((W + 2 * P - FW) / S + 1)
    return OH, OW

In [7]:
def init_conv_params(C: int,
                     FH: int,
                     FW: int,
                     FN: int,
                     rng: np.random.Generator) -> Tuple[np.ndarray, np.ndarray]:
    """初始化卷积层的权重(滤波器)和偏置

    Args:
    C (int): 输入数据的通道数
    FH (int): 滤波器的高度
    FW (int): 滤波器的宽度
    FN (int): 滤波器的数量 (也就是输出特征图的通道数)
    rng (np.random.Generator): NumPy 随机数生成器实例

    Returns:
    Tuple[np.ndarray, np.ndarray]:
    - W: 权重参数，形状为 (FN, C, FH, FW)。请按照正态分布随机生成，并乘以 0.01 缩小方差。
    - b: 偏置参数，形状为 (FN,)。请使用 np.zeros 初始化为 0。
    """

    W = rng.normal(0, 1, size=(FN, C, FH, FW)) * 0.01
    b = np.zeros((FN,))
    return W, b

In [8]:
def im2col(input_data, filter_h, filter_w, stride=1, pad=0):
    """
    将图像数据展开为二维矩阵的工具函数
    """
    N, C, H, W = input_data.shape
    out_h = (H + 2*pad - filter_h)//stride + 1
    out_w = (W + 2*pad - filter_w)//stride + 1

    img = np.pad(input_data, [(0,0), (0,0), (pad, pad), (pad, pad)], 'constant')
    col = np.zeros((N, C, filter_h, filter_w, out_h, out_w))

    for y in range(filter_h):
        y_max = y + stride*out_h
        for x in range(filter_w):
            x_max = x + stride*out_w
            col[:, :, y, x, :, :] = img[:, :, y:y_max:stride, x:x_max:stride]

    col = col.transpose(0, 4, 5, 1, 2, 3).reshape(N*out_h*out_w, -1)
    return col

In [9]:
def col2im(col: np.ndarray,
           input_shape: Tuple[int, int, int, int],
           filter_h: int, filter_w: int,
           stride: int = 1, pad: int = 0) -> np.ndarray:
    """将二维矩阵还原为图像形状的多维数组 (im2col 的逆操作)

    Args:
    col (np.ndarray): 展开的二维矩阵，形状为 (N * OH * OW, C * filter_h * filter_w)
    input_shape (Tuple[int, int, int, int]): 原始输入数据的形状 (N, C, H, W)
    filter_h (int): 滤波器的高度
    filter_w (int): 滤波器的宽度
    stride (int, optional): 步幅大小. 默认为 1.
    pad (int, optional): 填充大小. 默认为 0.

    Returns:
    np.ndarray: 还原后的多维数组，形状为 (N, C, H, W)
    """
    N, C, H, W = input_shape
    out_h = (H + 2*pad - filter_h) // stride + 1
    out_w = (W + 2*pad - filter_w) // stride + 1

    # 1. 将传入的 2D 梯度矩阵变回 6D 形状，准备按块还原
    col = col.reshape(N, out_h, out_w, C, filter_h, filter_w).transpose(0, 3, 4, 5, 1, 2)

    # 2. 创建一个带 padding 大小的全 0 空白画板
    img = np.zeros((N, C, H + 2*pad + stride - 1, W + 2*pad + stride - 1))

    # 3. 遍历滑动窗口的每一个位置，将梯度一块块“贴”回去
    for y in range(filter_h):
        y_max = y + stride * out_h
        for x in range(filter_w):
            x_max = x + stride * out_w
            # 【注意这里是 += ！】这就是我们刚才讨论的核心：重叠位置的梯度必须累加
            img[:, :, y:y_max:stride, x:x_max:stride] += col[:, :, y, x, :, :]

    # 4. 把周围一圈为了计算而加上的 Padding 切掉，返回原始图像大小
    return img[:, :, pad:H + pad, pad:W + pad]


* **输入数据 $x$**：形状为 `(N, C, H, W)` -> (批大小, 通道数, 高, 宽)
* **滤波器（权重）$W$**：形状为 `(FN, C, FH, FW)` -> (滤波器数量, 通道数, 滤波器高, 滤波器宽)

## 核心步骤

1. **计算输出尺寸** 📏
* 根据公式算出特征图的输出高 `OH` 和宽 `OW`。


2. **展开输入数据 (`im2col`)** 🖼️ ➡️ 📊
* **操作**：`col = im2col(x, FH, FW, stride, pad)`
* **形状**：变成二维矩阵 `(N * OH * OW, C * FH * FW)`。
* **含义**：行数代表**所有滑动窗口的总位置数**；列数代表**每个窗口被拉平后的元素总个数**。


3. **展开滤波器矩阵** 🍔 ➡️ 📊
* **操作**：`col_W = W.reshape(FN, -1).T`
* **形状**：变成二维矩阵 `(C * FH * FW, FN)`。
* **含义**：转置是为了让矩阵乘法的内侧维度对齐。


4. **矩阵乘法（点积）** ✖️
* **操作**：`out = np.dot(col, col_W) + b`
* **形状**：变成二维矩阵 `(N * OH * OW, FN)`。
* **含义**：算出每个局部位置上，所有滤波器产生的卷积结果。


5. **恢复空间结构 (`reshape`)** 📦
* **操作**：`out = out.reshape(N, OH, OW, FN)`
* **含义**：因为底层数据是按“图片->高->宽”的顺序存放的，直接切分就能完美复原出每个滑动位置的空间结构。


6. **调整维度顺序 (`transpose`)** 🔀
* **操作**：`out = out.transpose(0, 3, 1, 2)`
* **最终形状**：`(N, FN, OH, OW)`
* **含义**：把通道数（`FN`，原本在索引 3）移到前面，变回标准的 CNN 四维张量格式。

In [10]:
def conv_forward(x: np.ndarray, W: np.ndarray, b: np.ndarray, pad: int = 0, stride: int = 1) -> np.ndarray:
    """卷积层的前向传播处理

    Args:
    x (np.ndarray): 输入数据，形状为 (N, C, H, W)
    W (np.ndarray): 权重(滤波器)，形状为 (FN, C, FH, FW)
    b (np.ndarray): 偏置，形状为 (FN,)
    pad (int, optional): 填充大小. 默认为 0.
    stride (int, optional): 步幅大小. 默认为 1.

    Returns:
    np.ndarray: 卷积后的输出，形状为 (N, FN, OH, OW)
    """
    # 1. 获取形状参数
    FN, C, FH, FW = W.shape
    N, C, H, W_img = x.shape  # 使用 W_img 避免和权重 W 变量名冲突

    # 2. 计算出输出高度 OH 和 宽度 OW
    OH, OW = calculate_output_size(H, W_img, FH, FW, P=pad, S=stride)

    # 3. 用 im2col 展开输入 x
    col = im2col(x, FH, FW, stride, pad)

    # 4. 将滤波器 W 展开为二维矩阵
    w = W.reshape(FN, -1).T

    # 5. 计算点积并加上偏置 b
    out = col @ w + b

    # 6. 将得到的二维矩阵还原，将其转换为最终的 (N, FN, OH, OW) 形状 。
    return out.reshape(N, OH, OW, -1).transpose(0, 3, 1, 2)

In [11]:
def pool_forward(x: np.ndarray, pool_h: int, pool_w: int, pad: int = 0, stride: int = 1) -> np.ndarray:
    """池化层的前向传播处理 (Max Pooling)

    Args:
    x (np.ndarray): 输入数据，形状为 (N, C, H, W)
    pool_h (int): 池化窗口的高度
    pool_w (int): 池化窗口的宽度
    stride (int, optional): 步幅大小. 默认为 1.
    pad (int, optional): 填充大小. 默认为 0.

    Returns:
    np.ndarray: 池化后的输出，形状为 (N, C, OH, OW)
    """
    N, C, H, W = x.shape

    # 请在这里实现后续步骤：
    # 1. 计算输出的高度 OH 和 宽度 OW (复用 calculate_output_size)
    OH, OW = calculate_output_size(H, W, pool_h, pool_w, pad, stride)
    # 2. 用 im2col 展开 x
    col = im2col(x, pool_h, pool_w, stride, pad)
    # 3. 将展开后的矩阵 reshape 成 (-1, pool_h * pool_w)
    col = col.reshape(-1, pool_h * pool_w)
    # 4. 用 np.max 求每一行的最大值
    out = np.max(col, axis=1)
    # 5. 将结果 reshape 成 (N, OH, OW, C)，然后 transpose 成 (N, C, OH, OW) 并返回
    return out.reshape(N, OH, OW, -1).transpose(0, 3, 1, 2)

In [12]:
def relu_forward(x: np.ndarray) -> np.ndarray:
    """ReLU 激活函数前向传播

    Args:
    x (np.ndarray): 输入数据，形状可以是任意维度

    Returns:
    np.ndarray: 激活后的输出数据，形状与输入 x 相同。将小于 0 的元素变成 0，大于 0 的保持不变
    """
    return np.where(x < 0, 0, x)

In [13]:
def affine_forward(x: np.ndarray, W: np.ndarray, b: np.ndarray) -> np.ndarray:
    """全连接层 (Affine) 前向传播

    Args:
    x (np.ndarray): 输入数据，形状为 (N, d1, d2, ...)。注意：CNN中传入的x往往是四维的 (N, C, H, W)，需要先将其拉平成二维矩阵 (N, -1)
    W (np.ndarray): 权重矩阵，形状为 (D, out_size)，其中 D 是输入 x 拉平后的特征维度
    b (np.ndarray): 偏置向量，形状为 (out_size,)

    Returns:
    np.ndarray: 全连接层的输出，形状为 (N, out_size)
    """

    x = x.reshape(x.shape[0], -1)
    return  x @ W + b

relu的反向传播还是比较有意思的。大于0的导数为1，小于0的导数为0

In [14]:
def relu_backward(dout: np.ndarray, x: np.ndarray) -> np.ndarray:
    """ReLU 激活函数的反向传播

    Args:
    dout (np.ndarray): 上游传来的梯度，形状与 x 相同
    x (np.ndarray): 前向传播时的输入数据，用于判断哪些位置的元素大于 0

    Returns:
    np.ndarray: 传给下游的梯度 dx，形状与 x 相同
    """
    out = np.where(x < 0, 0, 1)
    return dout * out

In [15]:
def affine_backward(dout: np.ndarray, x: np.ndarray, W: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """全连接层 (Affine) 的反向传播

    Args:
    dout (np.ndarray): 上游传来的梯度，形状为 (N, out_size)
    x (np.ndarray): 前向传播时的输入数据，形状可能是四维 (N, C, H, W) 或二维 (N, D)
    W (np.ndarray): 权重矩阵，形状为 (D, out_size)

    Returns:
    Tuple[np.ndarray, np.ndarray, np.ndarray]:
    - dx: 传给下游的梯度，必须 reshape 回输入 x 的原始形状
    - dW: 权重的梯度，形状与 W 相同 (D, out_size)
    - db: 偏置的梯度，形状与 b 相同 (out_size,)
    """

    # dx 最后的形状和 x 相同
    dx_flat = dout @ W.T
    dx = dx_flat.reshape(x.shape)

    # 计算 dW 必须先把 x 拍平
    x_flat = x.reshape(x.shape[0], -1)
    dW = x_flat.T @ dout

    db = np.sum(dout, axis=0)

    return dx, dW, db

In [16]:
def conv_backward(dout: np.ndarray, x: np.ndarray, W: np.ndarray, stride: int = 1, pad: int = 0) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """卷积层的反向传播

    Args:
    dout (np.ndarray): 上游传来的梯度，形状为 (N, FN, OH, OW)
    x (np.ndarray): 前向传播时的输入数据，形状为 (N, C, H, W)
    W (np.ndarray): 权重(滤波器)，形状为 (FN, C, FH, FW)
    stride (int, optional): 步幅大小. 默认为 1.
    pad (int, optional): 填充大小. 默认为 0.

    Returns:
    Tuple[np.ndarray, np.ndarray, np.ndarray]:
    - dx: 传给下游的输入梯度，形状为 (N, C, H, W)
    - dW: 滤波器的权重梯度，形状为 (FN, C, FH, FW)
    - db: 偏置的梯度，形状为 (FN,)
    """

    FN, C, FH, FW = W.shape
    N, C, H, W_idx = x.shape

    # 1. 前向传播时的展开矩阵
    col = im2col(x, FH, FW, stride, pad)
    col_W = W.reshape(FN, -1).T

    # 2. 上游梯度变 2D
    dout_col = dout.transpose(0, 2, 3, 1).reshape(-1, FN)

    # -------------------------------------
    # 3. 你的推导成果：核心梯度计算！
    db = np.sum(dout_col, axis=0)
    dW_col = col.T @ dout_col
    dcol = dout_col @ col_W.T
    # -------------------------------------

    # 4. 将 2D 梯度恢复为原始的 4D 形状
    dW = dW_col.T.reshape(FN, C, FH, FW)
    dx = col2im(dcol, x.shape, FH, FW, stride, pad)

    return dx, dW, db

## 池的反向传播

池化主要是从 `x` 中挑选一些矩阵中最大的值，

它的反向传播的关键是将这些最大值位置的值进行保留并反向传播到上一层。而其他未选中的值则为 $0$

这里主要是三个步骤

1. 获取原始的 `x` 经过 `im2col` 处理后得到的最大值的位置(索引)。
2. 创建和 `x` 形状一致的全零矩阵 `dmax`，将 `dout` 拍平，然后把它的值存在 `dmax` 对应的位置。
3. 将 `dmax` 转换为和 `x` 相同的形状并返回。

In [17]:
def pool_backward(dout: np.ndarray, x: np.ndarray, pool_h: int, pool_w: int, stride: int = 1, pad: int = 0) -> np.ndarray:
    """池化层 (Max Pooling) 的反向传播

    Args:
    dout (np.ndarray): 上游传来的梯度，形状为 (N, C, OH, OW)
    x (np.ndarray): 前向传播时的输入数据，形状为 (N, C, H, W)
    pool_h (int): 池化窗口的高度
    pool_w (int): 池化窗口的宽度
    stride (int, optional): 步幅大小. 默认为 1.
    pad (int, optional): 填充大小. 默认为 0.

    Returns:
    np.ndarray: 传给下游的梯度 dx，形状与原始输入 x 相同 (N, C, H, W)
    """

    N, C, H, W = x.shape

    col = im2col(x, pool_h, pool_w, stride, pad)
    col = col.reshape(-1, pool_h * pool_w)
    mask = np.argmax(col, axis=1)

    dmax = np.zeros_like(col)
    dout_flat = dout.transpose(0, 2, 3, 1).flatten()
    row_idx = np.arange(dmax.shape[0])
    dmax[row_idx, mask] = dout_flat

    return col2im(dmax, x.shape, pool_h, pool_w, stride, pad)

## Softmax

它的标准数学公式是这样的：

$$y_k = \frac{e^{x_k}}{\sum_{i=1}^{n} e^{x_i}}$$

### 分子部分：$e^{x_k}$ (指数放大与正数化)

* **输入 $x_k$**：也就是上一层（全连接层）输出的原始得分，它可能是正数、负数甚至 0（比如 10, -2, 0）。
* **指数函数 $e^x$ 的作用**：
    * **保证非负**：概率不能是负数。无论 $x_k$ 多小，$e^{x_k}$ 永远大于 0。
    * **拉开差距**：指数函数的增长非常陡峭。如果一个类别的原始得分比其他类别稍微高一点点，经过 $e^x$ 放大后，它的值会变得**巨大无比**，从而在最终的概率里占据绝对统治地位。这就是它叫 "Soft**max**" 的原因——它是一种“平滑版”的取最大值操作。


### 分母部分：$\sum_{i=1}^{n} e^{x_i}$ (归一化)

* 这个部分就是把分子中计算出的所有类别的指数值，全部加起来求一个总和。
* **作用**：既然要把数值变成“概率”，那么所有类别的概率加起来必须等于 1（也就是 100%）。让每一个放大的得分（分子）去除以这个总和（分母），就完成了**归一化（Normalization）**，确保最后输出的 $y_1, y_2, \dots, y_n$ 的和等于 1。


### 致命缺陷：“溢出（Overflow）”

如果某个 $x_k$ 特别大（比如 1000），$e^{1000}$ 会导致计算机的浮点数直接爆炸（变成 `inf`），进而导致分式计算出 `NaN`（Not a Number）。

为了解决这个问题，我们在计算前，会给所有的 $x_i$ 统一减去一个常数 $C$。这不会改变最终的概率结果：

$$y_k = \frac{e^{x_k - C}}{\sum_{i=1}^{n} e^{x_i - C}}$$

在实际写代码时，为了达到最好的防溢出效果，我们通常把这个常数 $C$ 设为**输入数组 $x$ 中的最大值**（即 $C = \max(x)$）。这样一来，减去 $C$ 之后，数组里的最大值就变成了 0，所有的指数 $e^x$ 顶多就是 $e^0 = 1$，彻底消灭了溢出的可能，而那些原本很小的负数经过 $e^x$ 后会无限趋近于 0，完全不影响大局。


In [18]:
def softmax(x: np.ndarray) -> np.ndarray:
    """计算 Softmax 概率分布

    Args:
    x (np.ndarray): 网络的原始输出得分，形状为 (N, num_classes) 或 (num_classes,)

    Returns:
    np.ndarray: 经过指数归一化后的概率分布，形状与 x 相同
    """
    C = x.max(axis=-1, keepdims=True)
    numerator = np.exp(x - C)
    denominator = np.sum(numerator, axis=-1, keepdims=True)
    return numerator / denominator

In [19]:
x = np.array([1.0, 2.0, 3.0])
x2 = np.array([[1, 2, 3],[4, 5, 6]])
y = softmax(x2)

## 多分类交叉熵损失
$$L = - \sum_{k=1}^{K} t_k \ln(y_k)$$

In [20]:
def cross_entropy_error(y: np.ndarray, t: np.ndarray) -> float:
    """计算交叉熵误差

    Args:
    y (np.ndarray): 神经网络的输出概率分布，形状为 (N, num_classes)
    t (np.ndarray): 真实的标签数据 (One-hot 编码格式或标签索引格式)

    Returns:
    float: 计算出的平均交叉熵误差
    """
    delta = 1e-7

    # 对应 t 为标签索引格式
    if y.ndim == t.ndim + 1:
        batch_size = y.shape[0]
        # 直接用高级索引抽出正确类别的概率！t本身就是位置
        y_true_class = y[np.arange(batch_size), t]
        return -np.mean(np.log(y_true_class + delta))
    # 对应 t 为one-hot格式
    else:
        return - np.mean(np.sum(t * np.log(y+delta), axis=-1))

In [ ]:
def softmax_loss(x: np.ndarray, t: np.ndarray) -> Tuple[float, np.ndarray]:
    """计算 Softmax 概率、交叉熵误差以及梯度

    Args:
    x (np.ndarray): 网络最后一层全连接层的原始输出得分，形状为 (N, num_classes)
    t (np.ndarray): 真实的标签数据 (One-hot 编码格式或标签索引格式)

    Returns:
    Tuple[float, np.ndarray]:
    - loss: 计算出的交叉熵误差 (标量)
    - dx: 传给下游的梯度，形状为 (N, num_classes)
    """
    # 首先算出损失
    y = softmax(x)
    loss = cross_entropy_error(y, t)

    # 然后直接对它进行梯度下降。y - t
    t = to_categorical(t, y.shape[1]) \
        if y.ndim == t.ndim + 1 else t
    batch_size = y.shape[0]
    dx = (y - t) / batch_size

    return loss, dx

In [ ]:
def sgd(w: np.ndarray, dw: np.ndarray, lr: float) -> np.ndarray:
    """
    使用随机梯度下降 (SGD) 更新参数

    Args:
    w (np.ndarray): 当前的权重或偏置矩阵
    dw (np.ndarray): 反向传播计算出的对应梯度
    lr (float): 学习率 (Learning Rate)，控制迈步的大小

    Returns:
    np.ndarray: 更新后的新参数矩阵
    """

    updated_w = - w - dw * lr

    return updated_w

In [ ]:
def cnn_predict(x: np.ndarray, params: dict, conv_param: dict, pool_param: dict) -> np.ndarray:
    """简单 CNN 的推理 (前向传播) 过程

    网络结构: Conv -> ReLU -> Pooling -> Affine -> ReLU -> Affine

    Args:
    x (np.ndarray): 输入数据，形状为 (N, C, H, W)
    params (dict): 包含权重的字典，键包括 'W1', 'b1', 'W2', 'b2', 'W3', 'b3'
    conv_param (dict): 卷积超参数，例如 {'pad': 0, 'stride': 1}
    pool_param (dict): 池化超参数，例如 {'pool_h': 2, 'pool_w': 2, 'stride': 2, 'pad': 0}

    Returns:
    np.ndarray: 网络的原始输出得分，形状为 (N, out_size)
    """

    # 1. 提取参数
    W1, b1 = params['W1'], params['b1']
    W2, b2 = params['W2'], params['b2']
    W3, b3 = params['W3'], params['b3']

    # 前向传播：
    # 1. conv_forward
    conv_out = conv_forward(x, W1, b1, conv_param['pad'], conv_param['stride'])
    # 2. relu_forward
    relu1_out = relu_forward(conv_out)
    # 3. pool_forward
    pool_out = pool_forward(relu1_out,
                            pool_param['pool_h'], pool_param['pool_w'],
                            pool_param['pad'], pool_param['stride'])
    # 4. affine_forward (第一层全连接)
    affine1_out = affine_forward(pool_out, W2, b2)
    # 5. relu_forward
    relu2_out = relu_forward(affine1_out)
    # 6. affine_forward (第二层全连接)
    out = affine_forward(relu2_out, W3, b3)

    return out